# Visual Feature Extraction — MELD

All three visual streams share the same face-crop pipeline:
1. Sample frames at 2 fps (max 60)
2. RetinaFace detects face → crops once per frame
3. Valid crops split into temporal segments: beginning / middle / end (3 frames each)
4. All three models receive the same face crops

**Outputs** (per utterance, per split):
- `video_clip_temporal_{split}.pt`    — `{clip_name: np.array(3, 768)}`   CLIP ViT-L/14
- `video_siglip2_temporal_{split}.pt` — `{clip_name: np.array(3, 1152)}`  SigLIP 2
- `video_openface_au_{split}.pt`      — `{clip_name: np.array(3, 8)}`     AU intensities (temporal)

Zero vectors stored for clips where no face detected in any frame.

In [1]:
import os
import cv2
import tempfile
import torch
import numpy as np
import pandas as pd
import clip
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from transformers import AutoProcessor, AutoModel as HFAutoModel
from openface.face_detection import FaceDetector
from openface.multitask_model import MultitaskPredictor

In [2]:
PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/MELD"
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

FRAME_SAMPLE_FPS  = 2      # sample 2 frames/sec per clip
MAX_FRAMES        = 60     # cap — mirrors 30s audio truncation guard
N_FRAMES_PER_SEG  = 3      # frames per temporal segment
AU_CONF_THRESHOLD = 0.5    # RetinaFace vis_threshold
AU_DIM            = 8      # OpenFace 3.0 MTL outputs 8 AU intensities
CKPT_EVERY        = 500    # save checkpoint every N utterances
SPLITS            = ["train", "dev", "test"]
OF3_WEIGHTS_DIR   = Path.home() / ".cache" / "openface"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

Device   : cuda
GPU      : NVIDIA GeForce RTX 3060
VRAM free: 11.7 GB


In [3]:
labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Total utterances : {len(labels)}")
print(f"Split counts:\n{labels['split'].value_counts()}")

Total utterances : 13708
Split counts:
split
train    9989
test     2610
dev      1109
Name: count, dtype: int64


In [4]:
def sample_frames(video_path: Path, fps: float = FRAME_SAMPLE_FPS,
                  max_frames: int = MAX_FRAMES) -> list[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []
    src_fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
    interval = max(1, int(round(src_fps / fps)))
    frames, idx = [], 0
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret or frame is None or frame.size == 0:
            break
        if idx % interval == 0:
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def temporal_segments(items: list, n: int = N_FRAMES_PER_SEG) -> tuple:
    total  = len(items)
    mid    = total // 2
    half_n = n // 2
    beg     = items[:n]
    mid_seg = items[max(0, mid - half_n): max(0, mid - half_n) + n]
    end     = items[max(0, total - n):]
    return beg, mid_seg, end

## Load Models

In [5]:
clip_model, clip_preprocess = clip.load("ViT-L/14", device=DEVICE)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad_(False)
CLIP_DIM = clip_model.visual.output_dim
print(f"CLIP        : ViT-L/14  dim={CLIP_DIM}")

CLIP        : ViT-L/14  dim=768


In [6]:
siglip2_processor = AutoProcessor.from_pretrained("google/siglip2-so400m-patch14-384")
siglip2_model     = HFAutoModel.from_pretrained("google/siglip2-so400m-patch14-384")
siglip2_model.eval()
for p in siglip2_model.parameters():
    p.requires_grad_(False)
siglip2_model = siglip2_model.to(DEVICE)
SIGLIP2_DIM   = siglip2_model.config.vision_config.hidden_size
print(f"SigLIP 2    : so400m-patch14-384  dim={SIGLIP2_DIM}")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

SigLIP 2    : so400m-patch14-384  dim=1152


In [7]:
of3_face = FaceDetector(
    model_path=str(OF3_WEIGHTS_DIR / "Alignment_RetinaFace.pth"),
    device=str(DEVICE),
    vis_threshold=AU_CONF_THRESHOLD,
)
of3_pred = MultitaskPredictor(
    model_path=str(OF3_WEIGHTS_DIR / "MTL_backbone.pth"),
    device=str(DEVICE),
)
print(f"OpenFace 3.0: RetinaFace + MTL backbone  AU_DIM={AU_DIM}")

Loading pretrained model from /home/jubaer/.cache/openface/Alignment_RetinaFace.pth
remove prefix 'module.'
Missing keys:0
Unused checkpoint keys:0
Used keys:300
Loading multitask model from /home/jubaer/.cache/openface/MTL_backbone.pth...
OpenFace 3.0: RetinaFace + MTL backbone  AU_DIM=8


/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name tf_efficientnet_b0_ns to current tf_efficientnet_b0.ns_jft_in1k.
  model = create_fn(


## Extraction Functions

Face detection runs **once per clip**. Resulting crops are passed to all three models.


In [8]:
def collect_face_crops(video_path: Path) -> list[np.ndarray]:
    """Sample frames → RetinaFace → return list of valid BGR face crops."""
    frames = sample_frames(video_path)
    crops  = []
    with tempfile.TemporaryDirectory() as tmpdir:
        for i, frame in enumerate(frames):
            fpath = os.path.join(tmpdir, f"f{i:04d}.jpg")
            cv2.imwrite(fpath, frame)
            crop, _ = of3_face.get_face(fpath)
            if crop is not None and crop.size > 0:
                crops.append(crop)
    return crops


def encode_clip(crops: list[np.ndarray]) -> np.ndarray:
    """Face crops → temporal (3, CLIP_DIM)."""
    if not crops:
        return np.zeros((3, CLIP_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(crops):
        if not seg:
            vecs.append(np.zeros(CLIP_DIM, dtype=np.float32)); continue
        pil = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in seg]
        t   = torch.stack([clip_preprocess(p) for p in pil]).to(DEVICE)
        with torch.no_grad():
            e = clip_model.encode_image(t).float()
            e = e / e.norm(dim=-1, keepdim=True)
        vecs.append(e.cpu().numpy().mean(axis=0))
    return np.stack(vecs)   # (3, CLIP_DIM)


def encode_siglip2(crops: list[np.ndarray]) -> np.ndarray:
    """Face crops → temporal (3, SIGLIP2_DIM)."""
    if not crops:
        return np.zeros((3, SIGLIP2_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(crops):
        if not seg:
            vecs.append(np.zeros(SIGLIP2_DIM, dtype=np.float32)); continue
        pil    = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in seg]
        inputs = siglip2_processor(images=pil, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = siglip2_model.vision_model(**inputs)
        vecs.append(out.pooler_output.cpu().float().numpy().mean(axis=0))
    return np.stack(vecs)   # (3, SIGLIP2_DIM)


def encode_au(crops: list[np.ndarray]) -> np.ndarray:
    """Face crops → temporal (3, AU_DIM)."""
    if not crops:
        return np.zeros((3, AU_DIM), dtype=np.float32)
    vecs = []
    for seg in temporal_segments(crops):
        if not seg:
            vecs.append(np.zeros(AU_DIM, dtype=np.float32)); continue
        aus = []
        for crop in seg:
            _, _, au_out = of3_pred.predict(crop)
            aus.append(au_out.cpu().float().numpy().flatten())
        vecs.append(np.stack(aus).mean(axis=0))
    return np.stack(vecs)   # (3, AU_DIM)


print("Extraction functions defined.")

Extraction functions defined.


## Extraction — Unified Loop

> **Re-run note:** existing `.pt` files used full-frame CLIP/SigLIP2 and flat AU.
> Run the cell below to delete them before proceeding.

In [9]:
# Delete old .pt files so the extraction loop re-runs with new approach.
# Only run this cell once — after this, the loop below re-extracts everything.
# for split in SPLITS:
#     for tag in ["video_clip_temporal", "video_siglip2_temporal", "video_openface_au"]:
#         p = FEATURES_DIR / f"{tag}_{split}.pt"
#         if p.exists():
#             p.unlink()
#             print(f"Deleted: {p.name}")
# print("Done. Run the extraction cell next.")

In [10]:
for split in SPLITS:
    clip_path = FEATURES_DIR / f"video_clip_temporal_{split}.pt"
    sig_path  = FEATURES_DIR / f"video_siglip2_temporal_{split}.pt"
    au_path   = FEATURES_DIR / f"video_openface_au_{split}.pt"

    if clip_path.exists() and sig_path.exists() and au_path.exists():
        print(f"[{split}] all exist — skipping"); continue

    # Checkpoint files — deleted on successful completion
    ckpt_clip = FEATURES_DIR / f"video_clip_temporal_{split}_ckpt.pt"
    ckpt_sig  = FEATURES_DIR / f"video_siglip2_temporal_{split}_ckpt.pt"
    ckpt_au   = FEATURES_DIR / f"video_openface_au_{split}_ckpt.pt"

    # Resume from checkpoint if exists, else start fresh
    clip_feats: dict[str, np.ndarray] = torch.load(ckpt_clip, weights_only=False) if ckpt_clip.exists() else {}
    sig_feats:  dict[str, np.ndarray] = torch.load(ckpt_sig,  weights_only=False) if ckpt_sig.exists()  else {}
    au_feats:   dict[str, np.ndarray] = torch.load(ckpt_au,   weights_only=False) if ckpt_au.exists()   else {}
    already_done = set(clip_feats.keys())

    split_df = labels[labels["split"] == split].reset_index(drop=True)
    n_resume = len(already_done)
    print(f"\n--- {split}  ({len(split_df)} utterances, {n_resume} already done) ---")

    skipped:   list[str] = []
    zero_face: list[str] = []
    n_new = 0

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split):
        if row["clip_name"] in already_done:
            continue

        vid_path = DATA_ROOT / row["video_path"]
        if not vid_path.exists() or row["status"] != "ok":
            skipped.append(row["clip_name"]); continue

        crops = collect_face_crops(vid_path)

        if not crops:
            zero_face.append(row["clip_name"])
            clip_feats[row["clip_name"]] = np.zeros((3, CLIP_DIM),    dtype=np.float32)
            sig_feats [row["clip_name"]] = np.zeros((3, SIGLIP2_DIM), dtype=np.float32)
            au_feats  [row["clip_name"]] = np.zeros((3, AU_DIM),      dtype=np.float32)
        else:
            clip_feats[row["clip_name"]] = encode_clip(crops)
            sig_feats [row["clip_name"]] = encode_siglip2(crops)
            au_feats  [row["clip_name"]] = encode_au(crops)

        n_new += 1
        if n_new % CKPT_EVERY == 0:
            torch.save(clip_feats, ckpt_clip)
            torch.save(sig_feats,  ckpt_sig)
            torch.save(au_feats,   ckpt_au)
            print(f"  [ckpt] {len(clip_feats)} total  ({n_new} new this run)")

    # Save final files and remove checkpoints
    torch.save(clip_feats, clip_path)
    torch.save(sig_feats,  sig_path)
    torch.save(au_feats,   au_path)
    for ckpt in [ckpt_clip, ckpt_sig, ckpt_au]:
        if ckpt.exists():
            ckpt.unlink()

    n_ok = sum(1 for v in clip_feats.values() if np.abs(np.array(v)).sum() > 1e-6)
    print(f"Saved  : {split}  ({len(clip_feats)} entries)")
    print(f"Face coverage : {n_ok}/{len(clip_feats)}  ({n_ok/max(len(clip_feats),1)*100:.1f}%)")
    if skipped:   print(f"Skipped (missing): {len(skipped)}")
    if zero_face: print(f"No face detected : {len(zero_face)}")


--- train  (9989 utterances, 0 already done) ---


train:   0%|          | 0/9989 [00:00<?, ?it/s]

  [ckpt] 500 total  (500 new this run)
  [ckpt] 1000 total  (1000 new this run)
  [ckpt] 1500 total  (1500 new this run)
  [ckpt] 2000 total  (2000 new this run)
  [ckpt] 2500 total  (2500 new this run)
  [ckpt] 3000 total  (3000 new this run)
  [ckpt] 3500 total  (3500 new this run)
  [ckpt] 4000 total  (4000 new this run)
  [ckpt] 4500 total  (4500 new this run)
  [ckpt] 5000 total  (5000 new this run)
  [ckpt] 5500 total  (5500 new this run)
  [ckpt] 6000 total  (6000 new this run)
  [ckpt] 6500 total  (6500 new this run)
  [ckpt] 7000 total  (7000 new this run)
  [ckpt] 7500 total  (7500 new this run)
  [ckpt] 8000 total  (8000 new this run)
  [ckpt] 8500 total  (8500 new this run)
  [ckpt] 9000 total  (9000 new this run)
  [ckpt] 9500 total  (9500 new this run)
Saved  : train  (9988 entries)
Face coverage : 9958/9988  (99.7%)
Skipped (missing): 1
No face detected : 30

--- dev  (1109 utterances, 0 already done) ---


dev:   0%|          | 0/1109 [00:00<?, ?it/s]

  [ckpt] 500 total  (500 new this run)
  [ckpt] 1000 total  (1000 new this run)
Saved  : dev  (1108 entries)
Face coverage : 1105/1108  (99.7%)
Skipped (missing): 1
No face detected : 3

--- test  (2610 utterances, 0 already done) ---


test:   0%|          | 0/2610 [00:00<?, ?it/s]

  [ckpt] 500 total  (500 new this run)
  [ckpt] 1000 total  (1000 new this run)
  [ckpt] 1500 total  (1500 new this run)
  [ckpt] 2000 total  (2000 new this run)
  [ckpt] 2500 total  (2500 new this run)
Saved  : test  (2610 entries)
Face coverage : 2606/2610  (99.8%)
No face detected : 4


## 4. Verification

In [11]:
print("=== Verification ===\n")
expected = {"train": 9989, "dev": 1109, "test": 2610}

for split in SPLITS:
    exp = expected[split]
    print(f"[{split}]")
    for tag, shape in [
        (f"video_clip_temporal_{split}",    (3, CLIP_DIM)),
        (f"video_siglip2_temporal_{split}", (3, SIGLIP2_DIM)),
        (f"video_openface_au_{split}",      (3, AU_DIM)),
    ]:
        pt = FEATURES_DIR / f"{tag}.pt"
        if not pt.exists():
            print(f"  {tag}: MISSING"); continue
        d   = torch.load(pt, weights_only=False)
        v   = next(iter(d.values()))
        arr = np.stack(list(d.values()))
        ok  = np.array(v).shape == shape
        print(f"  {tag}: count={len(d)}/{exp}  shape={np.array(v).shape} "
              f"{'✓' if ok else '✗ expected '+str(shape)}  "
              f"nan={np.isnan(arr).any()}  inf={np.isinf(arr).any()}")
    print()

=== Verification ===

[train]
  video_clip_temporal_train: count=9988/9989  shape=(3, 768) ✓  nan=False  inf=False
  video_siglip2_temporal_train: count=9988/9989  shape=(3, 1152) ✓  nan=False  inf=False
  video_openface_au_train: count=9988/9989  shape=(3, 8) ✓  nan=False  inf=False

[dev]
  video_clip_temporal_dev: count=1108/1109  shape=(3, 768) ✓  nan=False  inf=False
  video_siglip2_temporal_dev: count=1108/1109  shape=(3, 1152) ✓  nan=False  inf=False
  video_openface_au_dev: count=1108/1109  shape=(3, 8) ✓  nan=False  inf=False

[test]
  video_clip_temporal_test: count=2610/2610  shape=(3, 768) ✓  nan=False  inf=False
  video_siglip2_temporal_test: count=2610/2610  shape=(3, 1152) ✓  nan=False  inf=False
  video_openface_au_test: count=2610/2610  shape=(3, 8) ✓  nan=False  inf=False

